# Batch Jump-Height Calculation

Loops over every pose-data text file in a folder, parses the raw MediaPipe
landmark stream into named columns, runs the jump-height calculation on each,
and collects the results into one table.

**Data format note:** the raw files are a flat stream of `x,y,z` triplets —
33 landmarks per frame, in MediaPipe BlazePose order, with **no headers**.
The loader below reshapes that into the named columns
(`LEFT_HIP_x`, `RIGHT_FOOT_INDEX_y`, ...) your function expects.

**Before running:** start Jupyter from your project root so
`from utilities.utils import ...` etc. resolve.


## 1. Imports

In [11]:
import glob
import os
import traceback

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

from utilities.utils import flip_axis, ts_jump_height
import utilities.PTM as PTM

%matplotlib inline

## 2. Configuration

In [12]:
DATA_DIR = "keypoints/cmj"      # <-- folder containing your text files
PATTERN  = "*.txt"           # <-- e.g. "*.txt" or "*.csv"
FPS      = 30                # <-- frame rate of the source video

files = sorted(glob.glob(os.path.join(DATA_DIR, PATTERN)))
print(f"Found {len(files)} file(s):")
for f in files:
    print("  ", f)

Found 64 file(s):
   keypoints/cmj\P03_CMJBL_FRONT.txt
   keypoints/cmj\P03_CMJBL_FRONT_world.txt
   keypoints/cmj\P03_CMJUL_FRONT.txt
   keypoints/cmj\P03_CMJUL_FRONT_world.txt
   keypoints/cmj\P04_CMJ_BL_FRONT.txt
   keypoints/cmj\P04_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P04_CMJ_UL_FRONT.txt
   keypoints/cmj\P04_CMJ_UL_FRONT_world.txt
   keypoints/cmj\P05_CMJ_BL_FRONT.txt
   keypoints/cmj\P05_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P05_CMJ_UL_FRONT.txt
   keypoints/cmj\P05_CMJ_UL_FRONT_world.txt
   keypoints/cmj\P06_CMJ_BL_FRONT.txt
   keypoints/cmj\P06_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P06_CMJ_UL_FRONT.txt
   keypoints/cmj\P06_CMJ_UL_FRONT_world.txt
   keypoints/cmj\P07_CMJ_BL_FRONT.txt
   keypoints/cmj\P07_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P07_CMJ_UL_FRONT.txt
   keypoints/cmj\P07_CMJ_UL_FRONT_world.txt
   keypoints/cmj\P08_CMJ_BL_FRONT.txt
   keypoints/cmj\P08_CMJ_BL_FRONT_world.txt
   keypoints/cmj\P08_CMJ_UL_FRONT.txt
   keypoints/cmj\P08_CMJ_UL_FRONT_world.txt
  

## 3. Loader - flat triplet stream -> named columns

The raw file has no headers: it's `x,y,z` for landmark 0, then landmark 1, ...
through landmark 32, then the next frame, and so on. We read every number,
reshape to `(frames, 33, 3)`, and label each landmark using MediaPipe's
standard BlazePose order. Delimiters between numbers (comma, space, newline)
don't matter - we strip them all.

In [13]:
POSE_LANDMARKS = [
    "NOSE", "LEFT_EYE_INNER", "LEFT_EYE", "LEFT_EYE_OUTER",
    "RIGHT_EYE_INNER", "RIGHT_EYE", "RIGHT_EYE_OUTER", "LEFT_EAR", "RIGHT_EAR",
    "MOUTH_LEFT", "MOUTH_RIGHT", "LEFT_SHOULDER", "RIGHT_SHOULDER",
    "LEFT_ELBOW", "RIGHT_ELBOW", "LEFT_WRIST", "RIGHT_WRIST",
    "LEFT_PINKY", "RIGHT_PINKY", "LEFT_INDEX", "RIGHT_INDEX",
    "LEFT_THUMB", "RIGHT_THUMB", "LEFT_HIP", "RIGHT_HIP",
    "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE", "RIGHT_ANKLE",
    "LEFT_HEEL", "RIGHT_HEEL", "LEFT_FOOT_INDEX", "RIGHT_FOOT_INDEX",
]
COORDS = ("x", "y", "z")
PER_FRAME = len(POSE_LANDMARKS) * len(COORDS)   # 33 * 3 = 99


def load_pose_file(path):
    """Parse a headerless MediaPipe landmark stream into a labelled DataFrame."""
    text = open(path).read().replace(",", " ")
    vals = np.array(text.split(), dtype=float)

    if vals.size % PER_FRAME != 0:
        raise ValueError(
            f"{vals.size} numbers is not a multiple of {PER_FRAME} "
            f"(remainder {vals.size % PER_FRAME}). Expected 33 landmarks x 3 "
            f"coords per frame - check the export."
        )

    frames = vals.reshape(-1, len(POSE_LANDMARKS), len(COORDS))
    cols = {f"{name}_{c}": frames[:, i, j]
            for i, name in enumerate(POSE_LANDMARKS)
            for j, c in enumerate(COORDS)}
    return pd.DataFrame(cols)

### Quick check on one file

Confirm the parse looks like a real person before running the batch.

In [14]:
if files:
    df = load_pose_file(files[0])
    print(os.path.basename(files[0]), "->", df.shape, "(frames, columns)")
    for lm in ["NOSE", "LEFT_SHOULDER", "LEFT_HIP", "RIGHT_HIP", "RIGHT_FOOT_INDEX"]:
        print(f"  {lm:18s} y[0] = {df[lm + '_y'][0]:+.3f}")

ValueError: could not convert string to float: '-'

## 4. World-vs-image landmark check  (IMPORTANT)

If these are `pose_world_landmarks`, the origin is pinned to the hip every
frame, so **the hip never moves** and gravity-based scaling can't work.
Jump height needs `pose_landmarks` (normalized image coords) or pixels.

Run this on a file where the person clearly jumped:
- `hip_y` **barely moves** (tiny std/range) -> world landmarks: re-export using
  `results.pose_landmarks` instead of `results.pose_world_landmarks`.
- `hip_y` **swings clearly** -> image coords, you're good.

In [ ]:
if files:
    df = load_pose_file(files[0])
    hip_y = (df["LEFT_HIP_y"] + df["RIGHT_HIP_y"]) / 2
    print("hip_y  min -> max :", round(hip_y.min(), 4), "->", round(hip_y.max(), 4))
    print("hip_y  range      :", round(hip_y.max() - hip_y.min(), 4))
    print("hip_y  std        :", round(hip_y.std(), 4))
    print("\n-> near-zero range/std means WORLD landmarks (re-export needed).")

## 5. Jump-height calculation

Same logic as your original, with `fps` actually passed through to
`PTM.mm_per_px` (yours hardcoded 30) and `verbose`/`plot` as arguments that
default off for the batch.

In [ ]:
def calculate_jump(data, fps=30, verbose=False, plot=False):
    hip_x = (data["LEFT_HIP_x"] + data["RIGHT_HIP_x"]) / 2
    hip_y = (data["LEFT_HIP_y"] + data["RIGHT_HIP_y"]) / 2
    hip = np.column_stack((hip_x, hip_y))

    toe = data[["RIGHT_FOOT_INDEX_x", "RIGHT_FOOT_INDEX_y"]].to_numpy()

    hip_flipped = flip_axis(hip.copy())
    toe_flipped = flip_axis(toe.copy())

    ts = savgol_filter(hip_flipped[:, 1], window_length=11, polyorder=2)
    mm_px = PTM.mm_per_px(ts, fps=fps, verbose=verbose, plot=plot)

    toe_y_mm = toe_flipped[:, 1] * mm_px
    return ts_jump_height(toe_y_mm, fps=fps)

## 6. Run over every file

In [ ]:
results = []

for path in files:
    name = os.path.basename(path)
    try:
        data = load_pose_file(path)
        jh = calculate_jump(data, fps=FPS, verbose=False, plot=False)
        results.append({"file": name, "frames": len(data), "jump_height": jh, "error": ""})
        print(f"OK   {name:40s}  jump_height = {jh}")
    except Exception as e:
        results.append({"file": name, "frames": np.nan, "jump_height": np.nan, "error": str(e)})
        print(f"FAIL {name:40s}  {e}")
        # traceback.print_exc()

print(f"\nProcessed {len(results)} file(s).")


Processed 0 file(s).


## 7. Results table

In [ ]:
df_results = pd.DataFrame(results)
df_results

""


## 8. Save results

In [ ]:
out_path = "jump_heights.csv"
df_results.to_csv(out_path, index=False)
print(f"Saved -> {out_path}")

Saved -> jump_heights.csv


## 9. Debug a single clip (optional)

Turn the gravity plot and verbose output back on for one file.

In [ ]:
debug_file = files[0] if files else None

if debug_file:
    data = load_pose_file(debug_file)
    jh = calculate_jump(data, fps=FPS, verbose=True, plot=True)
    print(f"{os.path.basename(debug_file)}: jump_height = {jh}")

In [15]:
raw = open(files[0]).read()
print("FIRST 300 CHARS:\n", repr(raw[:300]))
print("\nLAST 200 CHARS:\n", repr(raw[-200:]))

def _isfloat(t):
    try:
        float(t); return True
    except ValueError:
        return False

toks = raw.replace(",", " ").split()
bad = [(i, t) for i, t in enumerate(toks) if not _isfloat(t)]
print("\ntotal tokens:", len(toks))
print("bad tokens (first 20):", bad[:20])
print("unique bad values:", sorted(set(t for _, t in bad)))

# show context around the first bad token
if bad:
    i = bad[0][0]
    print("\ncontext around first bad token:", toks[max(0, i-4):i+5])

FIRST 300 CHARS:
 '0.5353298783302307,0.43739670515060425,-0.36792150139808655\n0.53727126121521,0.43143218755722046,-0.3438895344734192\n0.5397437810897827,0.4314664900302887,-0.34388792514801025\n0.5419885516166687,0.43148452043533325,-0.34388935565948486\n0.5274337530136108,0.4311678409576416,-0.34984302520751953\n0.522'

LAST 200 CHARS:
 '429718,0.1913321316242218\n0.4822010397911072,0.7548829317092896,0.2067655771970749\n0.552358090877533,0.772067666053772,0.01727989688515663\n0.4540952146053314,0.7713143229484558,0.038703564554452896\n-\n'

total tokens: 112200
bad tokens (first 20): [(99, '-'), (199, '-'), (299, '-'), (399, '-'), (499, '-'), (599, '-'), (699, '-'), (799, '-'), (899, '-'), (999, '-'), (1099, '-'), (1199, '-'), (1299, '-'), (1399, '-'), (1499, '-'), (1599, '-'), (1699, '-'), (1799, '-'), (1899, '-'), (1999, '-')]
unique bad values: ['-']

context around first bad token: ['0.002823194954544306', '0.44772377610206604', '0.7749595642089844', '-0.01936536654829979', '